> **Archived.** The v9-era training notebook from the old
> `splunk-soar-api` repo. Superseded by `../02-train-model.ipynb`;
> kept for reference only and not runnable as-is.


# Disposition router — training pipeline

Routes an alert to an **AI analyst** or a **human analyst**.

| | |
| --- | --- |
| Label | `y=1` needs a human (True Positive + False Positive) · `y=0` AI lane (Benign Positive) |
| Input | one column: the whole Splunk ES notable artifact as text |
| Model | TF-IDF word 1–2 grams → averaged `SGDClassifier` (`log_loss`, L2, `alpha=1e-5`) |
| Frozen as | `models/v9` — ROC 0.8523 · PR 0.6888 |

This notebook is the pipeline end to end: source → table → split → fit → evaluate
→ freeze. It imports the same modules `freeze.py` uses, so nothing here is a
reimplementation that can drift from what ships.

It lives here in `soar/`, next to the extract it reads. The pipeline code and the
frozen releases live in `ai-soc/machine-learning/`, which the next cell puts on
the import path. Run it with this folder's `.venv` kernel — it has `pandas`,
`numpy`, `pyarrow`, `scikit-learn` and `joblib` at the same versions the pipeline
was built with.

**It does not overwrite `models/v9`.** The freeze cell is off by default.

In [ ]:
# The pipeline modules live with the models, in ai-soc/machine-learning. This
# notebook sits in soar/ next to the extract it reads, so put that folder on the
# path rather than copying the code - a second copy would drift from what ships.
import sys
from pathlib import Path

ML = Path("/Users/mohammad.yekrangian/Projects/poc/ai-soc/machine-learning")
if not ML.is_dir():
    raise SystemExit(f"pipeline modules not found at {ML}")
sys.path.insert(0, str(ML))

import json, warnings

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

import dataset as DS        # prior-only feature engine + time_split
import dataset12 as D12     # the extract -> the table
import sgd_model as SM      # the stack
import metrics as MX        # zero_miss_point / coverage
import freeze as FZ         # release machinery + oof_stack

REBUILD = False   # True re-reads the 90 MB extract (~1 min); False loads the parquet
FREEZE  = False   # True cuts a NEW release into models/ - leaves v9 alone

print("modules:", ML)
print("source :", D12.SOURCE.name, "-", f"{D12.SOURCE.stat().st_size/1e6:.0f} MB")
print("table  :", D12.OUT.name, "exists" if D12.OUT.exists() else "MISSING - set REBUILD=True")
print("models :", FZ.MODELS)

## 1 · The source

One row per SOAR container, 12 months, three SOAR servers. Most rows carry no
usable verdict — heartbeats, duplicates, and containers nobody dispositioned.

In [ ]:
raw = pd.read_parquet(D12.SOURCE, columns=["server", "id", "disposition", "close_time"])
print(f"containers: {len(raw):,}   servers: {raw.server.nunique()}")

counts = raw.disposition.fillna("<never dispositioned>").replace("", "<blank>").value_counts()
keep = pd.Series({k: counts.get(k, 0) for k in D12.KEEP})
print(f"\nusable verdicts: {keep.sum():,}  ({keep.sum()/len(raw)*100:.1f}% of containers)\n")
counts.to_frame("rows").assign(kept=lambda d: d.index.isin(D12.KEEP))

`id` is **server-local** — 123,897 containers share only 104,628 distinct ids, so
`(server, id)` is the key. The numeric `tenant` is server-local too. Both traps
are handled in `dataset12.normalise()`.

In [ ]:
print(f"distinct id          : {raw.id.nunique():,}")
print(f"distinct (server, id): {len(raw.drop_duplicates(['server','id'])):,}   <- the real key")

## 2 · Which artifact is safe to read

This is the step that decides whether the model means anything. A container
carries three families of CEF artifact and **only one predates the analyst**.

| Artifact | Identified by | |
| --- | --- | --- |
| ES notable | `rule_id` present, no `soar_event` | the alert as it arrived |
| settings / metrics | carries `SA_DISPOSITION` | **that field is the label** |
| `soar_event=incident_closure` | | carries `closure_disposition` |
| Slack / escalation | `reply_count`, `snooze_count` | only exists if escalated |

Absence of `soar_event` is **not** the test — two of the three unsafe families
lack it as well. `D12.is_notable()` requires `rule_id` present *and*
`SA_DISPOSITION` absent.

In [ ]:
cef = pd.read_parquet(D12.SOURCE, columns=["CEF", "disposition"])
labelled = cef[cef.disposition.isin(D12.KEEP)]
sample = labelled.sample(2000, random_state=7)

families, leak = {}, []
for arr, dispo in zip(sample.CEF, sample.disposition):
    for rawart in arr:
        try:
            o = json.loads(rawart)
        except Exception:
            continue
        if "soar_event" in o:            fam = f"soar_event={o['soar_event']}"
        elif "SA_DISPOSITION" in o:      fam = "settings (SA_DISPOSITION)"
        elif "rule_id" in o:             fam = "ES notable"
        else:                            fam = "slack / escalation"
        families[fam] = families.get(fam, 0) + 1
        if fam.startswith("settings"):
            leak.append(o["SA_DISPOSITION"] == dispo)

print("artifact families per 2,000 containers:")
for k, v in sorted(families.items(), key=lambda kv: -kv[1]):
    print(f"  {k:32s} {v:6,d}")
print(f"\nsettings artifact's SA_DISPOSITION equals the label: {np.mean(leak)*100:.1f}% of the time")
print("-> reading it would be training on the answer")

## 3 · Build the table

`dataset12.build()` normalises the extract into the vocabulary `dataset.build()`
speaks, then attaches `text_cef`. Ordering is by **`close_time`** — when a human
actually ruled the container — not arrival, so the forward split is honest.

In [ ]:
if REBUILD:
    frame = D12.build()
    frame.to_parquet(D12.OUT, index=False)
else:
    frame = pd.read_parquet(D12.OUT)

print(f"rows {len(frame):,}   columns {len(frame.columns)}")
print(f"span {frame.t.min():%Y-%m-%d} -> {frame.t.max():%Y-%m-%d}")
print(f"human lane {frame.y_human.sum():,} ({frame.y_human.mean()*100:.1f}%)   "
      f"AI lane {(1-frame.y_human).sum():,} ({(1-frame.y_human).mean()*100:.1f}%)")
print(f"rules {frame.rule.nunique()}   customers {frame.tenant.nunique()}")

### The base rate drifts — badly

6.5% human-lane in 2025-11 against 25.3% in 2026-09. This is why thresholds are
cut from the whole year rather than a recent slice, and why anything the model
can read as a *date* has to go.

In [ ]:
by_month = (frame.assign(m=frame.t.dt.tz_convert(None).dt.to_period("M"))
            .groupby("m").y_human.agg(rows="size", human_pct="mean"))
by_month["human_pct"] = (by_month.human_pct * 100).round(1)
by_month

## 4 · The input column

Every field of the ES notable, flattened to `key value` text. Keys are kept so a
bigram can bind a field to its reading — `severity low` stays distinct from
`urgency low`.

In [ ]:
blob = frame.text_cef
print(f"non-empty {(blob.str.len() > 0).mean()*100:.1f}%   distinct {blob.nunique():,} of {len(blob):,}")
print(f"words per blob: median {blob.str.split().str.len().median():.0f}, "
      f"p90 {blob.str.split().str.len().quantile(.9):.0f}\n")
print(blob.iloc[5][:520])

### Timestamps are scrubbed, deliberately

The notable carries dates in ~20 fields and inside filenames and descriptions.
The first fit put `2026 04` among its strongest human-lane terms — a date
predicts the *training window* and nothing beyond it.

Scrubbing costs **0.0002 ROC**. The signal was never the clock.

In [ ]:
demo = ('user_startDate 1643818140.000000 firstSeen 2026-08-19T14:04:42.523Z '
        'fileName "Exported Spend 2026-04-13T1447.xlsx" login_time 21:04:15 severity high')
print("before:", demo)
print("after :", D12.scrub_time(demo))

## 5 · The forward split

Train on what came first, test on what came after — the way it deploys. Never
random: a random split would let a model see an alert's own future.

In [ ]:
train, test = DS.time_split(frame, holdout=0.30)
print(f"train {len(train):,} rows to {train.t.max():%Y-%m-%d}  ({train.y_human.mean()*100:.1f}% human)")
print(f"test  {len(test):,} rows from {test.t.min():%Y-%m-%d}  ({test.y_human.mean()*100:.1f}% human)")

## 6 · Fit

One column in, one probability out.

In [ ]:
model = SM.build(alpha=1e-5, average=True)
model.fit(SM.columns(train), train.y_human)

vec = model.named_steps["features"].named_transformers_["cef_word"]
X = model.named_steps["features"].transform(SM.columns(train))
print(f"vocabulary        {X.shape[1]:,} terms (word 1-2 grams, min_df=2, sublinear_tf)")
print(f"non-zeros per row {X.nnz/X.shape[0]:.0f}   density {X.nnz/(X.shape[0]*X.shape[1])*100:.3f}%")
print(f"columns consumed  {list(SM.columns(train).columns)}")

score_holdout = model.predict_proba(SM.columns(test))[:, 1]
print(f"\nholdout  ROC {roc_auc_score(test.y_human, score_holdout):.4f}  "
      f"PR {average_precision_score(test.y_human, score_holdout):.4f}")

## 7 · What it learned

`min_df=2` does real work here: an entity seen exactly once is discarded, so only
**recurring** users, hosts and subnets get a weight.

The bare numbers are IP octets — the word tokeniser splits `165.225.36.87` into
`165 225 36 87`, so `10 100` means *the 10.100.x.x subnet*. Crude tokenisation,
but it is genuinely how the signal arrives.

In [ ]:
w = model.named_steps["sgd"].coef_[0]
names = np.array(vec.get_feature_names_out())
order = np.argsort(w)
pd.DataFrame({
    "human-lane term": names[order[::-1][:15]],
    "weight":          w[order[::-1][:15]].round(3),
    "AI-lane term":    names[order[:15]],
    "weight ":         w[order[:15]].round(3),
})

## 8 · Out-of-fold — the evaluation that counts

A single holdout is one window and one draw. `FZ.oof_stack` scores every block
with a model fitted **only on what came before it**, across five expanding
forward folds, so the whole table gets an honest score.

This is the number in the manifest. Takes ~2 minutes.

In [ ]:
fit_on, features = FZ.make(alpha=1e-5)
scored, oof = FZ.oof_stack(frame, fit_on, features)
y = scored.y_human.to_numpy()

print(f"scored {len(y):,} rows / {y.sum():,} human-lane  (base {y.mean()*100:.2f}%)")
print(f"ROC {roc_auc_score(y, oof):.4f}   PR {average_precision_score(y, oof):.4f}")

### The operating table

Accuracy is the wrong metric in both directions — everything-to-AI scores 78.3%
while missing every real case; everything-to-human is the only safe policy and
scores 21.7%. The question is **how much of the queue the AI can absorb before it
starts closing real work.**

`residual` is the figure to put in front of whoever signs off: the rate at which
something genuine gets auto-closed.

In [ ]:
rows = []
for rate in (0, 0.25, 0.5, 1, 2, 3, 5, 8):
    budget = int(round(rate/100 * y.sum()))
    share, thr = MX.zero_miss_point(y, oof, budget)
    ai = oof < thr if share else np.zeros(len(y), bool)
    rows.append({"miss rate %": rate, "missed": budget, "threshold": round(thr, 6),
                 "AI takes %": round(share, 2), "AI lane rows": int(ai.sum()),
                 "residual %": round(y[ai].mean()*100 if ai.sum() else 0.0, 2)})
pd.DataFrame(rows).set_index("miss rate %")

In [ ]:
fifth = pd.qcut(oof, 5, labels=False, duplicates="drop")
pd.DataFrame({
    "predicted %": (pd.Series(oof).groupby(fifth).mean()*100).round(1),
    "actual %":    (pd.Series(y).groupby(fifth).mean()*100).round(1),
}).rename_axis("score fifth")

## 9 · Freeze

Writes `model.joblib`, `manifest.json` and a README into `models/vN`. The
manifest fingerprints the predictions, so anyone can prove the artifact they hold
is the one that was measured.

`--full` (here `full=True`) fits on every row and takes its thresholds from the
out-of-fold scores above — there is no held-out window left in a deployment
release, so the thresholds must come from somewhere honest.

**Set `FREEZE = True` in the setup cell to cut a release.** Off by default so
re-running this notebook does not litter `models/`.

In [ ]:
if FREEZE:
    path = FZ.freeze(full=True)
    print("frozen ->", path.relative_to(FZ.HERE))
    print(json.dumps(json.loads((path/"manifest.json").read_text())["metrics"]
                     | {}, indent=2)[:400])
else:
    print("FREEZE is False - nothing written.")
    print("the live release is:")
    man = json.loads((FZ.MODELS/"v9"/"manifest.json").read_text())
    print(f"  v9  ROC {man['metrics']['roc_auc']}  PR {man['metrics']['pr_auc']}")
    print(f"  input: {man['features']}")
    print(f"  frozen at {man['frozen_at']}")

## Known limits

- **The target is noisy.** Closing desks disagree by 15.3 points on the same rule
  at the same customer. That ceiling sits above anything the model can reach, and
  every architecture tried — TF-IDF, embeddings, gradient boosting, a fine-tuned
  ATTACK-BERT — lands in a 0.77–0.85 band beneath it. **Better labels beat better
  models here.**
- **It memorises entities.** Real usernames, emails and addresses carry weight.
  That is the source of the gain, and a PII consideration for a persisted
  artifact.
- **It needs the whole notable at inference**, absent on 4.9% of containers.
  Those score near the base rate and route to a human — correct, but it is a
  silent dependency.
- **The base rate drifts.** Thresholds sit between the 2025 and 2026 regimes;
  re-cut them if the mix moves again.